# Handling Ambiguity and Improving Clarity in Prompt Engineering

## Overview

This tutorial focuses on two critical aspects of prompt engineering: identifying and resolving ambiguous prompts, and techniques for writing clearer prompts. These skills are essential for effective communication with AI models and obtaining more accurate and relevant responses. Without a firm grasp of these fundamentals, even well-intentioned prompts can produce wildly inconsistent outputs — wasting time, resources, and undermining trust in AI systems. Understanding how to diagnose and fix ambiguity before it reaches the model is what separates novice prompt writers from skilled practitioners.

## Motivation

Ambiguity in prompts can lead to inconsistent or irrelevant AI responses, while lack of clarity can result in misunderstandings and inaccurate outputs. By mastering these aspects of prompt engineering, you can significantly improve the quality and reliability of AI-generated content across various applications. This matters because AI models do not infer intent the way humans do in conversation — they pattern-match against what is written. A vague prompt doesn't prompt the model to ask for clarification; it prompts the model to guess, and that guess may be confidently wrong. In production environments, this guessing behavior compounds across thousands of interactions, making systematic clarity not just a quality concern but a business-critical one.

## Key Components

1. **Identifying ambiguous prompts** — Ambiguity is often invisible to its author. Learning to spot it requires deliberately adopting the perspective of a system that has no shared context, assumptions, or background knowledge with you. This skill trains you to audit your own prompts before submission.
2. **Strategies for resolving ambiguity** — Knowing that a prompt is ambiguous is only half the battle. Having a concrete toolkit of resolution strategies — adding context, constraining scope, specifying format — means you can act on that diagnosis reliably and efficiently.
3. **Techniques for writing clearer prompts** — Proactive clarity is more powerful than reactive fixes. These techniques instill habits that prevent ambiguity from arising in the first place, leading to faster iteration cycles and more predictable model behavior.
4. **Practical examples and exercises** — Abstract principles without application tend not to stick. Hands-on exercises force you to engage with real failure cases and develop the muscle memory needed to apply these concepts under pressure.

## Method Details

We'll use OpenAI's GPT model and the LangChain library to demonstrate various techniques for handling ambiguity and improving clarity in prompts. The tutorial will cover:

1. **Setting up the environment and necessary libraries** — A consistent, reproducible environment ensures that differences in output stem from prompt changes and not from configuration drift, making it possible to isolate and study the exact effects of ambiguity.
2. **Analyzing ambiguous prompts and their potential interpretations** — By deliberately examining how a single prompt can be read in multiple valid ways, you build the critical eye needed to anticipate model misinterpretation before it happens.
3. **Implementing strategies to resolve ambiguity, such as providing context and specifying parameters** — Concrete implementation grounds the theory. Seeing exactly how adding a single sentence of context shifts model output makes the value of these strategies immediately tangible and measurable.
4. **Exploring techniques for writing clearer prompts, including using specific language and structured formats** — Structured formats like numbered instructions or role assignments reduce interpretive freedom in ways that are predictable and controllable, giving you a systematic lever for improving output quality.
5. **Practical exercises to apply these concepts in real-world scenarios** — Real-world scenarios introduce the messiness and constraint that synthetic examples omit. Working through them prepares you to handle the ambiguity that emerges organically in actual projects, not just in controlled demonstrations.

## Setup

First, let's import the necessary libraries and set up our environment.

In [8]:
import os
from dotenv import load_dotenv
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

In [2]:
# 1. Load the .env file
# This automatically puts variables into os.environ
load_dotenv() 

# 2. Check if the key exists (Fail Fast)
if not os.getenv("OPENAI_API_KEY"):
    raise ValueError("OPENAI_API_KEY is missing! Check your .env file.")

# 3. Initialize (LangChain automatically looks for the env var)
llm = ChatOpenAI(model="gpt-4o-mini")

## Identifying Ambiguous Prompts

Let's start by examining some ambiguous prompts and analyzing their potential interpretations.

In [3]:
ambiguous_prompts = [
    "Tell me about the bank.",
    "What's the best way to get to school?",
    "Can you explain the theory?"
]

for prompt in ambiguous_prompts:
    analysis_prompt = f"Analyze the following prompt for ambiguity: '{prompt}'. Explain why it's ambiguous and list possible interpretations."
    print(f"Prompt: {prompt}")
    print(llm.invoke(analysis_prompt).content)
    print("-" * 50)

Prompt: Tell me about the bank.
The prompt "Tell me about the bank." is ambiguous for several reasons:

1. **Type of Bank**: The term "bank" can refer to different types of financial institutions, such as:
   - A commercial bank that provides checking and savings accounts, loans, and other financial services.
   - An investment bank that deals with capital markets, assists in mergers and acquisitions, or helps clients raise capital.
   - A central bank, which manages a country's currency, monetary policy, and financial system.

2. **Location**: The prompt does not specify which bank or where the bank is located. "The bank" could refer to a specific well-known bank (like JPMorgan Chase or Bank of America) or to a local bank that the speaker has in mind.

3. **Context of Inquiry**: The key focus of the inquiry is unclear:
   - Are they asking for general information about how banks operate?
   - Are they interested in specific services provided by the bank in question?
   - Do they want 

## Resolving Ambiguity

Now, let's explore strategies for resolving ambiguity in prompts.

In [4]:
from typing import List, Dict

def resolve_ambiguity(prompt: str, context: str) -> str:
    """
    Resolves ambiguity using structured chat roles.
    """
    # Messages are structured as a list of dictionaries (standard OpenAI/LangChain format)
    messages = [
        {"role": "system", "content": f"System Instruction: {context}"},
        {"role": "user", "content": prompt}
    ]
    
    # Modern LLM wrappers usually take the list directly
    response = llm.invoke(messages)
    return response.content

In [5]:
# Example usage
ambiguous_prompt = "Tell me about the bank."
contexts = [
    "You are a financial advisor discussing savings accounts.",
    "You are a geographer describing river formations."
]

for ctx in contexts:
    print(f"\n--- Acting as: {ctx} ---")
    print(resolve_ambiguity(ambiguous_prompt, ctx))


--- Acting as: You are a financial advisor discussing savings accounts. ---
When discussing banks, it's important to consider various aspects, including the types of accounts they offer, their interest rates, fees, and services. Here are some key points about banks, particularly in the context of savings accounts:

1. **Types of Accounts**: Banks typically offer a variety of savings accounts, such as standard savings accounts, high-yield savings accounts, money market accounts, and certificates of deposit (CDs). Each type has different features, interest rates, and minimum balance requirements.

2. **Interest Rates**: Savings accounts generally earn interest on the balance you maintain. Traditional savings accounts may offer lower rates, while high-yield savings accounts often provide more competitive interest rates, especially those offered by online banks.

3. **Fees**: Be sure to review any fees associated with a savings account, such as monthly maintenance fees or fees for falling

## Techniques for Writing Clearer Prompts

Let's explore some techniques for writing clearer prompts to improve AI responses.

In [7]:
def compare_prompt_clarity(original_prompt, improved_prompt):
    """
    Compare the responses to an original prompt and an improved, clearer version.
    
    Args:
    original_prompt (str): The original, potentially unclear prompt
    improved_prompt (str): An improved, clearer version of the prompt
    
    Returns:
    tuple: Responses to the original and improved prompts
    """
    original_response = llm.invoke(original_prompt).content
    improved_response = llm.invoke(improved_prompt).content
    return original_response, improved_response

# Example usage
original_prompt = "How do I make it?"
improved_prompt = "Provide a step-by-step guide for making a classic margherita pizza, including ingredients and cooking instructions."

original_response, improved_response = compare_prompt_clarity(original_prompt, improved_prompt)

print("Original Prompt Response:")
print(original_response)
print("\nImproved Prompt Response:")
print(improved_response)

Original Prompt Response:
Could you please provide more details about what you're trying to make? Whether it's a recipe, a craft project, or something else entirely, I'll be happy to help!

Improved Prompt Response:
Here's a step-by-step guide for making a classic Margherita pizza, including both the ingredients you'll need and detailed cooking instructions.

### Ingredients

**For the Dough:**
- 2 ¼ teaspoons active dry yeast (1 packet)
- 1 teaspoon sugar
- ¾ cup warm water (110°F or 43°C)
- 2 cups all-purpose flour (plus extra for dusting)
- 1 teaspoon salt
- 1 tablespoon olive oil (plus extra for greasing)

**For the Toppings:**
- 1 cup tomato sauce (preferably San Marzano tomatoes, crushed)
- 8 oz fresh mozzarella cheese (preferably buffalo mozzarella), sliced into rounds
- Fresh basil leaves
- Extra virgin olive oil
- Salt, to taste

### Equipment
- Mixing bowl
- Baking sheet or pizza stone
- Rolling pin
- Oven

### Instructions

#### Step 1: Prepare the Dough
1. **Activate the Ye

## Structured Prompts for Clarity

Using structured prompts can significantly improve clarity and consistency in AI responses.

In [9]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

# 1. Use ChatPromptTemplate for better multi-modal/chat model support
# 2. Use a single curly brace and format the list via the input logic
prompt = ChatPromptTemplate.from_template(
    """Provide an analysis of {topic} considering these key aspects:
{aspects}

Present the analysis in a {tone} tone."""
)

# Example usage
inputs = {
    "topic": "the impact of social media on society",
    "aspects": ["communication patterns", "mental health", "information spread"],
    "tone": "balanced and objective"
}

# 3. Modern pipe syntax with a string output parser
# We format the list into a string during the invoke step
chain = prompt | llm | StrOutputParser()

# We format the 'aspects' list into a bulleted string on the fly
formatted_aspects = "\n".join(f"- {a}" for a in inputs["aspects"])

response = chain.invoke({**inputs, "aspects": formatted_aspects})

print(response)

### Analysis of the Impact of Social Media on Society

Social media has profoundly transformed the way individuals interact, access information, and understand the world around them. While it has fostered greater connectivity and democratized information dissemination, it has also introduced significant challenges, particularly concerning communication patterns, mental health, and the spread of information. Below is a balanced examination of these key aspects.

#### 1. Communication Patterns

**Positive Impact:**
Social media platforms have revolutionized communication, allowing individuals to connect across vast distances, breaking geographical barriers. This facilitation of instant communication enables families, friends, and communities to stay in touch more easily than ever before. Moreover, social media has created new forms of social interaction, such as group chats, live-streaming events, and virtual gatherings, enhancing the richness of interpersonal communication.

**Negative 